# Support Vector Machine:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Importing Dataset:

In [2]:
data = pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM']),
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

X_train = preprocessing.fit_transform(X_train)
X_test = preprocessing.transform(X_test)

In [6]:
param = {'kernel': ['linear', 'rbf', 'poly', 'sigmoid']}

grid = GridSearchCV(
    estimator=SVR(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")

Best Parameters: {'kernel': 'linear'}


In [7]:
param = {'kernel': ['linear'],
        'C' : [0.01, 0.1, 1, 10, 100, 1000]}

grid = GridSearchCV(
    estimator=SVR(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")

Best Parameters: {'C': 1000, 'kernel': 'linear'}


In [8]:
svm = SVR(kernel='linear', C=1000)

start_time = time.time()
svm.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 3.25s


## Results:

In [9]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [10]:
# Results on Training Data:

y_pred_train = svm.predict(X_train)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 77.17%

mean absolute error(Training Data): 19.18%

adjusted r2 Score: 77.10%


In [11]:
# Results on Testing Data:

y_pred_test = svm.predict(X_test)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 77.93%

mean absolute error (Testing Data): 20.53%

adjusted r2 Score: 77.87%


## Cross-Val-Score:

In [12]:
preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM']),
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

svm_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', SVR(kernel='linear', C=1000))
])

In [13]:
print(f"r2 Score: {np.mean(cross_val_score(estimator=svm_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 76.75%


In [14]:
print(f"mean absolute error: {-np.mean(cross_val_score(estimator=svm_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

mean absolute error: 19.85%


---

# Conclusion:

- *SVM Model* achieved a mean cross-validated **r2-Score = 76.75% & M.A.P.E = 19.85%**.
- Compared with KNN, SVM achieved lower r2 score **(76.75% vs 79.95%)** but a very similar MAPE **(19.85% vs 18.80%)**.